This file reads the raw time-series data from financial_data.db to generates  windowed samples with **10-day Stride** for training the encoder.

Each sample represents a single data point that contains four component tensors. For any given "event" date (T-1):
*   **global_stock** (100x5): The past 100 days (T-130 to T-31) of OHLCV data. This is the long-term context.
*   **local_stock** (30x20): The recent 30 days (T-30 to T-1) of the full 20-feature vector. This is the primary input to the encoder.
*   **local_macro** (30x20): The corresponding 30 days QQQ full features, used to provide macro context.
*   **label** (5x5): The future 5 days (T to T+4) of OHLCV data. This is the target for pre-training.

Normalization:

All price-based features (OHLCVs) are normalized relative to p0_local—the 'open' price on day T-30 (the first day of the local window).

Non-price features (Volume, RSI, MACD, etc.) in local_stock / local_macro are Z-score normalized based on their own 30-day window.

Saved samples can be found in the folder **processed_samples**

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import torch
import os
from tqdm.notebook import tqdm
from google.colab import drive

drive.mount('/content/drive')

pd.options.mode.chained_assignment = None

Mounted at /content/drive


In [ ]:
DB_PATH = '/content/drive/MyDrive/financial_data.db'
OUTPUT_DIR = '/content/drive/MyDrive/processed_samples/'

# Window Sizes
GLOBAL_WINDOW_SIZE = 100
LOCAL_WINDOW_SIZE = 30
LABEL_WINDOW_SIZE = 5

STRIDE = 10 # i.e. Create a new sample every 10 days
TEST_SET_RATIO = 0.20


MACRO_SYMBOL = 'QQQ'


GLOBAL_FEATURES = [
    'open', 'high', 'low', 'close', 'volume'
]

LOCAL_FEATURES = [
    'open', 'high', 'low', 'close', 'volume',
    'sma_5', 'sma_20', 'ema_5', 'ema_20',
    'macd', 'macd_signal', 'macd_hist',
    'rsi_14', 'atr_14',
    'dow_cos', 'dow_sin',
    'dom_cos', 'dom_sin',
    'moy_cos', 'moy_sin'
]

LABEL_FEATURES = [
    'open', 'high', 'low', 'close', 'volume'
]

# Features to be normalized using relative to reference p0
PRICE_NORM_COLS = [
    'open', 'high', 'low', 'close',
    'sma_5', 'sma_20', 'ema_5', 'ema_20'
]

# Features to be normalized using Z-score
ZSCORE_NORM_COLS = [
    'volume', 'macd', 'macd_signal', 'macd_hist', 'atr_14', 'rsi_14'
]

In [ ]:
"""
Utilities

"""
def list_tables(db_path):
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [table[0] for table in cursor.fetchall()]
    return tables

def load_stock_data(db_path, stock_symbol):
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql(f"SELECT * FROM {stock_symbol}", conn, index_col='timestamp')
        df.index = pd.to_datetime(df.index)
    return df

def normalize_price_window(window_df, ref_price, cols):
    """
    Normalizes price-based columns relative to a p0.

    """
    normalized_df = window_df.copy()
    for col in cols:
        if col in normalized_df.columns:
            normalized_df[col] = (normalized_df[col] / ref_price) - 1
    return normalized_df

def normalize_zscore_window(window_df, cols):
    """
    Calculates Z-score for window columns

    """
    normalized_df = window_df.copy()
    for col in cols:
        if col in normalized_df.columns:
            mean = normalized_df[col].mean()
            std = normalized_df[col].std()
            epsilon = 1e-8
            normalized_df[col] = (normalized_df[col] - mean) / (std + epsilon)
    return normalized_df

In [ ]:
def create_and_save_samples(db_path, output_dir, stock_symbols):
    print("--- Starting Sample Creation Pipeline ---")

    all_samples_dir = os.path.join(output_dir, 'all_samples')
    os.makedirs(all_samples_dir, exist_ok=True)

    try:
        macro_df = load_stock_data(db_path, MACRO_SYMBOL)
        print(f"Successfully loaded macro data for {MACRO_SYMBOL}.")
    except Exception as e:
        print(f"Error loading macro data for {MACRO_SYMBOL}: {e}")
        return []

    all_sample_files = []
    total_sample_size = GLOBAL_WINDOW_SIZE + LOCAL_WINDOW_SIZE + LABEL_WINDOW_SIZE

    for symbol in tqdm(stock_symbols, desc="Processing Stocks"):
        if symbol == MACRO_SYMBOL:
            continue # Skip the macro symbol itself

        try:
            stock_df = load_stock_data(db_path, symbol)
        except Exception as e:
            print(f"Could not load data for {symbol}. Error: {e}")
            continue

        print(f"\nProcessing {symbol}...")


        for i in range(0, len(stock_df) - total_sample_size + 1, STRIDE):
            global_stock_window_raw = stock_df.iloc[i : i + GLOBAL_WINDOW_SIZE]
            local_stock_window_raw = stock_df.iloc[i + GLOBAL_WINDOW_SIZE : i + GLOBAL_WINDOW_SIZE + LOCAL_WINDOW_SIZE]
            label_window_raw = stock_df.iloc[i + GLOBAL_WINDOW_SIZE + LOCAL_WINDOW_SIZE : i + total_sample_size]

            start_date = local_stock_window_raw.index.min()
            end_date = local_stock_window_raw.index.max()
            local_macro_window_raw = macro_df.loc[start_date:end_date]

            if not (len(global_stock_window_raw) == GLOBAL_WINDOW_SIZE and \
                 len(local_stock_window_raw) == LOCAL_WINDOW_SIZE and \
                 len(local_macro_window_raw) == LOCAL_WINDOW_SIZE and \
                 len(label_window_raw) == LABEL_WINDOW_SIZE):
                continue # Skip if there's a date mismatch, e.g., holidays

            p0_global = global_stock_window_raw['open'].iloc[0]
            global_norm = normalize_price_window(global_stock_window_raw, p0_global, ['open', 'high', 'low', 'close'])
            global_norm = normalize_zscore_window(global_norm, ['volume'])

            p0_local = local_stock_window_raw['open'].iloc[0]
            local_stock_norm = normalize_price_window(local_stock_window_raw, p0_local, PRICE_NORM_COLS)
            local_stock_norm = normalize_zscore_window(local_stock_norm, ZSCORE_NORM_COLS)

            p0_macro = local_macro_window_raw['open'].iloc[0]
            local_macro_norm = normalize_price_window(local_macro_window_raw, p0_macro, PRICE_NORM_COLS)
            local_macro_norm = normalize_zscore_window(local_macro_norm, ZSCORE_NORM_COLS)

            label_norm = normalize_price_window(label_window_raw, p0_local, ['open', 'high', 'low', 'close', 'volume'])

            global_tensor = torch.tensor(global_norm[GLOBAL_FEATURES].values, dtype=torch.float32)
            local_stock_tensor = torch.tensor(local_stock_norm[LOCAL_FEATURES].values, dtype=torch.float32)
            local_macro_tensor = torch.tensor(local_macro_norm[LOCAL_FEATURES].values, dtype=torch.float32)
            label_tensor = torch.tensor(label_norm[LABEL_FEATURES].values, dtype=torch.float32)


            sample_data = {
                'global_stock': global_tensor,
                'local_stock': local_stock_tensor,
                'local_macro': local_macro_tensor,
                'label': label_tensor,
                'metadata': {
                    'stock_symbol': symbol,
                    'local_start_date': start_date.strftime('%Y-%m-%d'),
                    'p0_local': p0_local 
                }
            }

            # Naming: TICKER_YYYY-MM-DD.pt
            filename = f"{symbol}_{start_date.strftime('%Y-%m-%d')}.pt"
            filepath = os.path.join(all_samples_dir, filename)
            torch.save(sample_data, filepath)
            all_sample_files.append(filepath)

    return all_sample_files

In [ ]:
def split_and_move_files(all_sample_files, output_dir, test_ratio):
    """
    Split files into train and test sub-directories.

    """
    print("\n--- Performing Train/Test Split ---")

    all_sample_files.sort(key=lambda x: os.path.basename(x).split('_')[1].replace('.pt', ''))

    split_index = int(len(all_sample_files) * (1 - test_ratio))
    train_files = all_sample_files[:split_index]
    test_files = all_sample_files[split_index:]

    train_dir = os.path.join(output_dir, 'train')
    test_dir = os.path.join(output_dir, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # Move files
    for f in tqdm(train_files, desc="Moving train files"):
        os.rename(f, os.path.join(train_dir, os.path.basename(f)))

    for f in tqdm(test_files, desc="Moving test files"):
        os.rename(f, os.path.join(test_dir, os.path.basename(f)))
    try:
        os.rmdir(os.path.join(output_dir, 'all_samples'))
    except OSError as e:
        print(f"Could not remove 'all_samples' directory: {e}")


    print(f"\nTotal samples: {len(all_sample_files)}")
    print(f"Training samples: {len(train_files)}")
    print(f"Testing samples: {len(test_files)}")

In [ ]:
# Run the Pipeline

stock_symbols = list_tables(DB_PATH)
print(f"Found tables for stocks: {stock_symbols}")


all_files = create_and_save_samples(DB_PATH, OUTPUT_DIR, stock_symbols)

if all_files:
    split_and_move_files(all_files, OUTPUT_DIR, TEST_SET_RATIO)
    print("\n--- Pipeline Complete! Samples are in train/test directories. ---")
else:
    print("\n--- No files were created. ---")

Found tables for stocks: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AVGO', 'COST', 'NFLX', 'AMD', 'ADBE', 'INTC', 'CSCO', 'TMUS', 'QCOM', 'CMCSA', 'MU', 'INTU', 'AMAT', 'GILD', 'ASML', 'LRCX', 'PANW', 'VRTX', 'REGN', 'CRWD', 'AMGN', 'SBUX', 'PEP', 'QQQ']
--- Starting Sample Creation Pipeline ---
Successfully loaded macro data for QQQ.


Processing Stocks:   0%|          | 0/31 [00:00<?, ?it/s]


Processing AAPL...

Processing MSFT...

Processing NVDA...

Processing AMZN...

Processing GOOGL...

Processing META...

Processing TSLA...

Processing AVGO...

Processing COST...

Processing NFLX...

Processing AMD...

Processing ADBE...

Processing INTC...

Processing CSCO...

Processing TMUS...

Processing QCOM...

Processing CMCSA...

Processing MU...

Processing INTU...

Processing AMAT...

Processing GILD...

Processing ASML...

Processing LRCX...

Processing PANW...

Processing VRTX...

Processing REGN...

Processing CRWD...

Processing AMGN...

Processing SBUX...

Processing PEP...

--- Finished Creating All Samples ---

--- Performing Train/Test Split ---


Moving train files:   0%|          | 0/2232 [00:00<?, ?it/s]

Moving test files:   0%|          | 0/558 [00:00<?, ?it/s]


Total samples: 2790
Training samples: 2232
Testing samples: 558

--- Pipeline Complete! Samples are in train/test directories. ---
